# Traducción y Análisis de Sentimiento con ML

En las lecciones anteriores aprendimos a construir un bot básico con `TextBlob`. Ahora vamos a explorar:

1. **Traducción automática** — convertir texto de un idioma a otro
2. **Análisis de sentimiento** — determinar si un texto es positivo, negativo o neutral

Ambas técnicas son la base de herramientas como Google Translate y los análisis de opiniones.

## El problema de la traducción

Traducir no es solo reemplazar palabras. Cada idioma tiene una estructura gramatical diferente.

### Ejemplo: Inglés → Irlandés

| Inglés | Irlandés | Literal |
|--------|----------|--------|
| I feel happy | Tá athas orm | Happy is upon me |

En irlandés, las emociones se expresan como algo que **está sobre ti**, no algo que **sientes**. Un traductor literal haría "Mise bhraitheann athas" ("me feel happy") — gramaticalmente incorrecto.

### Enfoque tradicional vs Machine Learning

- **Tradicional**: Reglas formales de gramática → identificar palabras → traducir cada una
- **ML**: Usar millones de traducciones humanas para detectar patrones y predecir la mejor traducción

Google Translate y otros usan el enfoque ML con enormes corpus bilingües.

## Ejercicio 1: Traducción con deep-translator

Usamos `deep-translator` porque la función `translate()` de TextBlob fue eliminada en versiones recientes (Google bloqueó el acceso no oficial).

Probemos a traducir la famosa primera línea de *Orgullo y Prejuicio* de Jane Austen.

In [ ]:
from deep_translator import GoogleTranslator

# Crear traductor (inglés → francés)
traductor = GoogleTranslator(source='en', target='fr')

# La famosa primera línea de Orgullo y Prejuicio
original = "It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife!"

traduccion = traductor.translate(original)

print("Original (inglés):")
print(original)
print()
print("Traducción (francés):")
print(traduccion)

### ¿Cómo funciona?

Detrás de escena, `deep-translator` usa **Google Translate** — un modelo de ML entrenado con millones de frases bilingües. No hay reglas manuales; el modelo aprendió a traducir por patrones.

La traducción es bastante buena: *"C'est une vérité universellement reconnue, qu'un homme célibataire en possession d'une bonne fortune doit avoir besoin d'une femme!"*

Incluso es más precisa que la traducción humana de 1932 de V. Leconte y Ch. Pressoir, que agregaba palabras innecesarias.

In [ ]:
# Probemos con otros idiomas
text = "I feel happy"

print(f"Original: {text}")
print()

# Traducir a diferentes idiomas
idiomas = {
    'es': 'Español',
    'de': 'Alemán',
    'ja': 'Japonés',
    'pt': 'Portugués',
    'it': 'Italiano'
}

for codigo, nombre in idiomas.items():
    traductor = GoogleTranslator(source='en', target=codigo)
    traduccion = traductor.translate(text)
    print(f"{nombre:12s}: {traduccion}")

### ¿Por qué funciona mejor que traducir palabra por palabra?

Si tradujéramos "I feel happy" palabra por palabra:
- "I" = Yo
- "feel" = siento
- "happy" = feliz

Pero en español decimos "**Estoy** feliz" o "**Me siento** feliz", no "Yo siento feliz".

Google Translate **aprende la estructura** de cada idioma con millones de ejemplos.

---

## Análisis de Sentimiento

El sentimiento mide qué tan **positivo** o **negativo** es un texto.

- **Polaridad**: de -1 (muy negativo) a +1 (muy positivo)
- **Subjetividad**: de 0 (objetivo/hecho) a 1 (subjetivo/opinión)

### El problema del sarcasmo

Considera esta frase: *"Great, that was a wonderful waste of time, I'm glad we are lost on this dark road"*

Un algoritmo simple detecta: great (+), wonderful (+), glad (+), waste (-), lost (-), dark (-)

Resultado: sentimiento mixto. Pero un humano entiende que es **sarcasmo negativo**.

El ML puede mejorar esto si entrena con suficientes ejemplos de sarcasmo.

## Ejercicio 2: Sentimiento en Orgullo y Prejuicio

Analicemos el sentimiento de dos frases del libro.

In [ ]:
from textblob import TextBlob

# Dos frases de Orgullo y Prejuicio
quote1 = "It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife."

quote2 = "Darcy, as well as Elizabeth, really loved them; and they were both ever sensible of the warmest gratitude towards the persons who, by bringing her into Derbyshire, had been the means of uniting them."

# Analizar sentimiento
sentiment1 = TextBlob(quote1).sentiment
sentiment2 = TextBlob(quote2).sentiment

print(f"Frase 1: {quote1[:50]}...")
print(f"  Polaridad: {sentiment1.polarity:.3f} | Subjetividad: {sentiment1.subjectivity:.3f}")
print()
print(f"Frase 2: {quote2[:50]}...")
print(f"  Polaridad: {sentiment2.polarity:.3f} | Subjetividad: {sentiment2.subjectivity:.3f}")

### ¿Qué vemos?

- **Frase 1**: Polaridad baja (0.21) — es una observación general, no una opinión personal
- **Frase 2**: Polaridad alta (0.70) y subjetividad alta (0.80) — habla de amor y gratitud, claramente emocional

Esto tiene sentido: la primera frase es una declaración "universal", la segunda expresa sentimientos personales.

## Ejercicio 3: Pesos de las palabras

Veamos el peso real de cada palabra en el diccionario de TextBlob.

In [ ]:
# Ejemplos con el peso real de cada palabra
ejemplos = [
    'I love this amazing day',
    'I hate this terrible day',
    'The weather is fine',
    'This is boring and dull',
    'Great wonderful fantastic',
    'Bad ugly horrible',
    'I feel happy',
    'I feel sad',
]

for texto in ejemplos:
    blob = TextBlob(texto)
    print(f'Frase: "{texto}"')
    print(f'  Polaridad total: {blob.polarity:.3f}')
    
    # Mostrar peso de cada palabra
    for palabra in texto.split():
        w = TextBlob(palabra)
        print(f'    "{palabra}" = {w.polarity:+.2f}')
    print()

### Diccionario de pesos

| Palabra | Peso | Significado |
|---------|------|-------------|
| `wonderful` | **+1.0** | Máximo positivo |
| `amazing` | **+0.6** | Muy positivo |
| `love` | **+0.5** | Positivo |
| `happy` | **+0.8** | Muy positivo |
| `fine` | **+0.42** | Moderadamente positivo |
| `horrible` | **-1.0** | Máximo negativo |
| `boring` | **-1.0** | Máximo negativo |
| `hate` | **-0.8** | Muy negativo |
| `sad` | **-0.5** | Negativo |

**Las palabras sin peso** (I, this, is, the, and) = 0.0 — son stop words o pronombres.

## Ejercicio 4: ¿Entiende el código las palabras?

Analicemos frases que son **positivas** o **negativas** para ver si el algoritmo acierta.

In [ ]:
# Frases claramente positivas
positive_phrases = [
    "I am so happy!",
    "This is delightful indeed!",
    "How wonderfully these sort of things occur!",
    "Charlotte is an excellent manager, I dare say.",
]

# Frases claramente negativas
negative_phrases = [
    "Everybody is disgusted with his pride.",
    "The pause was to Elizabeth's feelings dreadful.",
    "It would be dreadful!",
    "I have the greatest dislike in the world to that sort of thing.",
]

print("=== Frases POSITIVAS ===")
for phrase in positive_phrases:
    s = TextBlob(phrase).sentiment
    print(f"  [{s.polarity:+.2f}] {phrase}")

print()
print("=== Frases NEGATIVAS ===")
for phrase in negative_phrases:
    s = TextBlob(phrase).sentiment
    print(f"  [{s.polarity:+.2f}] {phrase}")

### Frases "trampa" — sarcasmo y contexto

Jane Austen es maestra del sarcasmo. Algunas frases que parecen positivas pero no lo son:

In [ ]:
# Estas frases tienen polaridad positiva pero en contexto son negativas
tricky_phrases = [
    "Happy shall I be, when his stay at Netherfield is over!",  # No es feliz, es sarcástica
    "If I could but see you as happy!",  # Expresa preocupación, no alegría
    "Our distress, my dear Lizzy, is very great.",  # Suena positivo pero es angustia
]

print("=== Frases 'trampa' ===")
for phrase in tricky_phrases:
    s = TextBlob(phrase).sentiment
    print(f"  [{s.polarity:+.2f}] {phrase}")
    print(f"         ¿El algoritmo acierta? {'Sí' if s.polarity < 0 else 'No — detecta positivo pero es sarcasmo/negativo'}")
    print()

### ¿Por qué falla el algoritmo?

1. **No entiende el contexto**: "Happy" aparece en la frase, pero la estructura "Happy shall I be when..." implica impaciencia
2. **No detecta sarcasmo**: "distress" es negativo, pero "my dear" y "very great" confunden al algoritmo
3. **Las palabras individuales engañan**: el análisis de sentimiento palabra por palabra falla con construcciones complejas

**Lección clave**: El NLP basado en palabras tiene limitaciones. Para sarcasmo y ironía, necesitamos modelos más sofisticados.

## Ejercicio 5: Análisis de sentimiento de un texto largo

Ahora vamos a analizar el sentimiento de frases con polaridad absoluta (1 o -1).

In [ ]:
# Texto de ejemplo (primeros párrafos de Orgullo y Prejuicio)
book_text = """
It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.
However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered the rightful property of some one or other of their daughters.
My dear Mr. Bennet, said his lady to him one day, have you heard that Netherfield Park is let at last?
Mr. Bennet replied that he had not.
But it is, returned she; for Mrs. Long has just been here, and she told me all about it.
Mr. Bennet made no answer.
Do you not want to know who has taken it? cried his wife impatiently.
You want to tell me, and I have no objection to hearing it.
This was invitation enough.
Why, my dear, you must know, Mrs. Long says that Netherfield is taken by a young man of large fortune from the north of England.
He came down on Tuesday in a chaise and four, and was wonderfully taken with the house.
"""

# Crear TextBlob y analizar cada oración
blob = TextBlob(book_text)

positive_sentences = []
negative_sentences = []

print("=== Análisis de sentimiento por oración ===")
for i, sentence in enumerate(blob.sentences):
    sentiment = sentence.sentiment
    polarity = sentiment.polarity
    
    if polarity == 1.0:
        positive_sentences.append(str(sentence))
    elif polarity == -1.0:
        negative_sentences.append(str(sentence))
    
    print(f"  [{polarity:+.2f}] {str(sentence)[:60]}...")

print(f"\nFrases con polaridad absoluta positiva (1.0): {len(positive_sentences)}")
print(f"Frases con polaridad absoluta negativa (-1.0): {len(negative_sentences)}")

---

## Resumen: Lo que aprendimos

| Concepto | Herramienta | Qué hace | Limitación |
|----------|-------------|----------|------------|
| **Traducción** | `deep-translator` | Traduce usando ML (Google Translate) | Necesita internet |
| **Polaridad** | `TextBlob` | Mide positivo/negativo (-1 a +1) | No entiende sarcasmo |
| **Subjetividad** | `TextBlob` | Mide hecho/opinión (0 a 1) | Depende del vocabulario |
| **Diccionario de pesos** | `TextBlob` | Cada palabra tiene un valor numérico | No entiende contexto |

### ¿Por qué importa?

- **Empresas**: Analizar miles de reseñas automáticamente
- **Política**: Clasificar emails de ciudadanos a favor/en contra
- **Marketing**: Medir percepción de marca en redes sociales

### El límite actual

El análisis de sentimiento basado en palabras funciona bien para texto directo, pero falla con:
- Sarcasmo e ironía
- Contexto cultural
- Estructuras gramaticales complejas

En la próxima lección veremos análisis de sentimiento más avanzado con **NLTK VADER**, que maneja mejor estos casos.

---

## Preguntas para reflexionar

1. ¿Por qué "wonderful" tiene peso +1.0 pero "fantastic" solo +0.4? ¿Quién decidió esos pesos?

2. Si un tweet dice "¡Qué gran servicio! Espero no volver nunca más" — ¿cómo determinarías el sentimiento real?

3. ¿Podría un modelo de sentimiento ser utilizado para manipular opiniones? ¿Qué ética tiene esto?